In [ ]:
# E7 = E6 (E3b + our PubChem channel, append-only) + our Class-3 channel (rr_bt) at ranks 4-8. E7 input: shishiradhikari11/casmi-e7-c3assets
# (our MIT code; assets derived from the competition train structures and COCONUT, CC BY 4.0). Inputs: competition data, prvsiyan/casmi26-fp-models-v2 (CC0),
# prvsiyan/casmi26-ranker-features (CC0), megayak/casmi26-simulated-ranker-rows (CC0),
# prvsiyan/coconut-casmi26-candidates (CC BY 4.0), shishiradhikari11/casmi-bio-clean (CC BY 4.0), RDKit wheel (BSD).
import os, sys, glob, re, subprocess, time, json, hashlib, pickle
import numpy as np, pandas as pd
T0 = time.time()
os.makedirs('/kaggle/working/numba_cache', exist_ok=True)
os.environ['NUMBA_CACHE_DIR'] = '/kaggle/working/numba_cache'
def find(pattern):
    hits = sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True), key=len)
    if not hits:
        raise FileNotFoundError(pattern)
    return hits[0]
tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
whl = [w for w in glob.glob('/kaggle/input/**/rdkit-*.whl', recursive=True) if tag in w]
print('rdkit wheel:', whl)
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--no-deps', whl[0]], capture_output=True, text=True)
print(r.stdout[-500:], r.stderr[-500:])
import rdkit; print('rdkit', rdkit.__version__, '' if rdkit.__version__.startswith(('2026.03', '2025.03')) else '!!! UNEXPECTED RDKIT VERSION !!!')
COMP = os.path.dirname(find('train.parquet'))
print('COMP', COMP, '| BIO', find('bio_fp.npy'), '| fp models', sorted(glob.glob('/kaggle/input/**/fp_*.pt', recursive=True)))

# E3 additions: shishiradhikari11/casmi-fm-runner (ms-pred MIT code + MassSpecGym weights, our runner and shims, RDKit 2025.3.6 wheel BSD).
SMOKE_N = 0            # > 0: engine on the first N molecules only (debug)
FM_ENABLE = True
FM_BUDGET_SEC = 5400     # wall clock for GLACIER + ICEBERG together (installation and model loading included)
FM_TOPN = 60                 # same-formula groups are formed inside the first 60 candidates
LAM_ICE, LAM_GL = 0.5, 0.5
FM_PROTECT_LIB = 0.6         # candidates with their own library similarity >= this keep their slot (None = E3 behaviour)
print('E3b switches: SMOKE_N', SMOKE_N, '| FM_BUDGET_SEC', FM_BUDGET_SEC, '| FM_TOPN', FM_TOPN, '| lam', LAM_ICE, LAM_GL, '| protect lib >=', FM_PROTECT_LIB)
try:
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())
except Exception as e:
    print('nvidia-smi:', repr(e))


In [ ]:
# Our engine (two-ranker + BIO + AFIX) in a subprocess -> ENG (E1 lists; E3 also exports scores, lib_max, top 60).
ENG_DIR = '/kaggle/working/eng'; os.makedirs(ENG_DIR, exist_ok=True)
ENG_FILES = {'pv.py': '"""prvsiyan analog-propagation pipeline components (public notebook, version of 2026-09-16), copied\nverbatim where possible so local simulations and the Kaggle notebook share one source of truth.\nAdditions are marked `# ADDED`.\n"""\nimport math\nimport numpy as np\nfrom numba import njit, prange\n\n\nclass CFG:\n    PPM_WIN = 10.0\n    PPM_FALLBACK = 30.0\n    INT_FLOOR = 0.002\n    MAX_PEAKS = 256\n    MZ_TOL = 0.01\n    INT_POWER = 1.0\n    ENT_WEIGHT = True\n    ANALOG_WIN = 200.0\n    N_ANALOG = 80\n    SIM_POWER = 3.0\n    W1_PRIORS = (0.30, 0.60)\n    SEEDS = (0, 1, 2, 3)\n    GBM = dict(max_depth=6, max_iter=500, learning_rate=0.03, min_samples_leaf=80, l2_regularization=1.0)\n    TOPN = 25\n\n\n# ----------------------------------------------------------------------------------------------\n# similarity kernels\n# ----------------------------------------------------------------------------------------------\n@njit(cache=True, fastmath=True)\ndef _clean(mz, it, floor, topk, power, ent_weight):\n    n = len(mz)\n    if n == 0: return np.empty(0, np.float32), np.empty(0, np.float32)\n    mx = 0.0\n    for i in range(n):\n        if it[i] > mx: mx = it[i]\n    if mx <= 0: return np.empty(0, np.float32), np.empty(0, np.float32)\n    thr = floor * mx; c = 0\n    for i in range(n):\n        if it[i] >= thr: c += 1\n    idx = np.empty(c, np.int64); j = 0\n    for i in range(n):\n        if it[i] >= thr: idx[j] = i; j += 1\n    if c > topk:\n        v = np.empty(c, np.float32)\n        for i in range(c): v[i] = it[idx[i]]\n        o = np.argsort(v)[c - topk:]\n        k2 = np.empty(topk, np.int64)\n        for i in range(topk): k2[i] = idx[o[i]]\n        k2.sort(); idx = k2; c = topk\n    om = np.empty(c, np.float32); oi = np.empty(c, np.float32)\n    s = 0.0\n    for i in range(c):\n        om[i] = mz[idx[i]]; v = it[idx[i]] ** power; oi[i] = v; s += v\n    if s > 0:\n        for i in range(c): oi[i] /= s\n    if ent_weight:\n        S = 0.0\n        for i in range(c):\n            if oi[i] > 0: S -= oi[i] * np.log(oi[i])\n        if S < 3.0:\n            w = 0.25 + 0.25 * S; s2 = 0.0\n            for i in range(c): oi[i] = oi[i] ** w; s2 += oi[i]\n            if s2 > 0:\n                for i in range(c): oi[i] /= s2\n    return om, oi\n\n\n@njit(cache=True, fastmath=True)\ndef entropy_sim(qmz, qp, cmz, cp, tol):\n    i = 0; j = 0; n = len(qmz); m = len(cmz)\n    SA = 0.0\n    for x in range(n):\n        if qp[x] > 0: SA -= qp[x] * np.log(qp[x])\n    SB = 0.0\n    for x in range(m):\n        if cp[x] > 0: SB -= cp[x] * np.log(cp[x])\n    SAB = 0.0; tot = 0.0\n    buf = np.empty(n + m, np.float64); b = 0\n    while i < n and j < m:\n        d = qmz[i] - cmz[j]\n        if d < -tol: buf[b] = qp[i]; i += 1; b += 1\n        elif d > tol: buf[b] = cp[j]; j += 1; b += 1\n        else: buf[b] = qp[i] + cp[j]; i += 1; j += 1; b += 1\n    while i < n: buf[b] = qp[i]; i += 1; b += 1\n    while j < m: buf[b] = cp[j]; j += 1; b += 1\n    for x in range(b): tot += buf[x]\n    if tot <= 0: return 0.0\n    for x in range(b):\n        v = buf[x] / tot\n        if v > 0: SAB -= v * np.log(v)\n    return 1.0 - (2.0 * SAB - SA - SB) / np.log(4.0)\n\n\n@njit(cache=True, fastmath=True, parallel=True)\ndef search(qmz, qp, cand, off, allmz, allin, tol, floor, topk, power, ent_weight):\n    out = np.zeros(len(cand), np.float32)\n    for k in prange(len(cand)):\n        c = cand[k]; a = off[c]; b = off[c + 1]\n        if b <= a: continue\n        cm, cp = _clean(allmz[a:b], allin[a:b], floor, topk, power, ent_weight)\n        if len(cm) == 0: continue\n        out[k] = entropy_sim(qmz, qp, cm, cp, tol)\n    return out\n\n\n@njit(cache=True, fastmath=True)\ndef entropy_sim_shift(qmz, qp, cmz, cp, tol, shift):\n    a = entropy_sim(qmz, qp, cmz, cp, tol)\n    if shift > -0.001 and shift < 0.001: return a\n    sm = np.empty(len(cmz), np.float32)\n    for i in range(len(cmz)): sm[i] = cmz[i] + shift\n    b = entropy_sim(qmz, qp, sm, cp, tol)\n    return a if a > b else b\n\n\n# ADDED: search over a pre-cleaned store (reps cleaned once instead of on every call)\n@njit(cache=True, fastmath=True, parallel=True)\ndef search_shift_pre(qmz, qp, lo, hi, roff, rmz, rit, tol, shift):\n    out = np.zeros(hi - lo, np.float32)\n    for k in prange(hi - lo):\n        a = roff[lo + k]; b = roff[lo + k + 1]\n        if b <= a: continue\n        out[k] = entropy_sim_shift(qmz, qp, rmz[a:b], rit[a:b], tol, shift[k])\n    return out\n\n\n# ADDED: library search where each reference is shifted by (query precursor - reference precursor):\n# the same molecule measured as a different adduct keeps its neutral losses (modified-cosine idea)\n@njit(cache=True, fastmath=True, parallel=True)\ndef search_shift_rows(qmz, qp, cand, off, allmz, allin, tol, floor, topk, power, ent_weight, shift):\n    out = np.zeros(len(cand), np.float32)\n    for k in prange(len(cand)):\n        c = cand[k]; a = off[c]; b = off[c + 1]\n        if b <= a: continue\n        cm, cp = _clean(allmz[a:b], allin[a:b], floor, topk, power, ent_weight)\n        if len(cm) == 0: continue\n        out[k] = entropy_sim_shift(qmz, qp, cm, cp, tol, shift[k])\n    return out\n\n\n# ADDED: clean a list of spectra into a flat store\n@njit(cache=True)\ndef clean_store(rows, off, allmz, allin, floor, topk, power, ent_weight):\n    n = len(rows)\n    lens = np.zeros(n + 1, np.int64)\n    tmp_m = []\n    tmp_i = []\n    for k in range(n):\n        r = rows[k]\n        cm, cp = _clean(allmz[off[r]:off[r + 1]], allin[off[r]:off[r + 1]], floor, topk, power, ent_weight)\n        tmp_m.append(cm); tmp_i.append(cp)\n        lens[k + 1] = lens[k] + len(cm)\n    om = np.empty(lens[n], np.float32); oi = np.empty(lens[n], np.float32)\n    for k in range(n):\n        om[lens[k]:lens[k + 1]] = tmp_m[k]; oi[lens[k]:lens[k + 1]] = tmp_i[k]\n    return lens, om, oi\n\n\n# ----------------------------------------------------------------------------------------------\n# adducts\n# ----------------------------------------------------------------------------------------------\nMASS = dict(C=12.0, H=1.00782503207, N=14.0030740048, O=15.9949146196, P=30.97376163,\n            S=31.97207100, F=18.99840322, Cl=34.96885268, Br=78.9183371, I=126.904473,\n            Na=22.9897692809, K=38.96370668, Si=27.9769265325, B=11.0093054, Se=79.9165213)\nE = 0.00054857990; PROTON = MASS[\'H\'] - E; H2O = 2 * MASS[\'H\'] + MASS[\'O\']\nNH4 = MASS[\'N\'] + 4 * MASS[\'H\']; FORMATE = MASS[\'C\'] + 2 * MASS[\'H\'] + 2 * MASS[\'O\']\nACETATE = 2 * MASS[\'C\'] + 4 * MASS[\'H\'] + 2 * MASS[\'O\']\nADDUCTS = {\n    "[M+H]+": (1, 1, PROTON), "[M+NH4]+": (1, 1, NH4 - E), "[M+Na]+": (1, 1, MASS[\'Na\'] - E),\n    "[M+K]+": (1, 1, MASS[\'K\'] - E), "[M-H2O+H]+": (1, 1, PROTON - H2O), "[M-2H2O+H]+": (1, 1, PROTON - 2 * H2O),\n    "[M+2H]2+": (1, 2, 2 * PROTON), "[M]+": (1, 1, -E), "[M-H2O]+": (1, 1, -E - H2O),\n    "[M+CH3OH+H]+": (1, 1, PROTON + MASS[\'C\'] + 4 * MASS[\'H\'] + MASS[\'O\']),\n    "[M+CH3CN+H]+": (1, 1, PROTON + 2 * MASS[\'C\'] + 3 * MASS[\'H\'] + MASS[\'N\']),\n    "[M-H]-": (1, 1, -PROTON), "[M-H2O-H]-": (1, 1, -PROTON - H2O), "[M+CH2O2-H]-": (1, 1, FORMATE - PROTON),\n    "[M+C2H4O2-H]-": (1, 1, ACETATE - PROTON), "[M+Cl]-": (1, 1, MASS[\'Cl\'] + E), "[M]-": (1, 1, E),\n    "[M-2H]-": (1, 2, -2 * PROTON), "[M+Na-2H]-": (1, 1, MASS[\'Na\'] - 2 * PROTON),\n    "[2M+H]+": (2, 1, PROTON), "[2M+Na]+": (2, 1, MASS[\'Na\'] - E), "[2M+NH4]+": (2, 1, NH4 - E),\n    "[2M+K]+": (2, 1, MASS[\'K\'] - E), "[2M-H]-": (2, 1, -PROTON), "[2M+CH2O2-H]-": (2, 1, FORMATE - PROTON),\n    "[2M+C2H4O2-H]-": (2, 1, ACETATE - PROTON), "[2M+Na-2H]-": (2, 1, MASS[\'Na\'] - 2 * PROTON),\n    "[3M+H]+": (3, 1, PROTON), "[3M-H]-": (3, 1, -PROTON),\n}\n\n\ndef neutral_mass(mz, adduct):\n    out = np.full(len(mz), np.nan); ad = np.asarray(adduct, dtype=object)\n    for a, (n, z, d) in ADDUCTS.items():\n        m = (ad == a)\n        if m.any(): out[m] = (mz[m] * z - d) / n\n    return out\n\n\n# ----------------------------------------------------------------------------------------------\n# ranking features (31)\n# ----------------------------------------------------------------------------------------------\nN_ANALOG = 80\nP_SIM = 3.0\nN_FEAT = 31\n\n\ndef _rank_norm(x):\n    o = np.argsort(-x); r = np.empty(len(x)); r[o] = np.arange(len(x)); return r / max(1, len(x) - 1)\n\n\ndef _z(x):\n    s = x.std()\n    return (x - x.mean()) / s if s > 1e-9 else np.zeros_like(x)\n\n\ndef rank_features(cand_fp, cand_lib, analog_fp, analog_sim, model_logits=None, frag=None):\n    nc = cand_fp.shape[0]\n    cf = cand_fp.astype(np.float32); cs = cf.sum(1)\n    lv = np.asarray(cand_lib, np.float32)\n    lvmax = float(lv.max()) if nc else 0.0\n    if analog_fp is not None and len(analog_sim):\n        af = analog_fp.astype(np.float32); asum = af.sum(1)\n        inter = cf @ af.T\n        tan = inter / (cs[:, None] + asum[None, :] - inter + 1e-9)\n        w = np.clip(np.asarray(analog_sim, np.float32), 0, None)\n        ap = (tan * (w ** P_SIM)[None, :]).max(1)\n        a1 = (tan * w[None, :]).max(1)\n        best_tan = tan.max(1); top_tan = tan[:, 0]; top_sim = float(w[0])\n        mean_tan = (tan * (w ** P_SIM)[None, :]).sum(1) / ((w ** P_SIM).sum() + 1e-9)\n    else:\n        ap = a1 = best_tan = top_tan = mean_tan = np.zeros(nc, np.float32); top_sim = 0.0\n    apmax = float(ap.max()) if nc else 0.0\n    if model_logits is not None:\n        raw = cf @ np.asarray(model_logits, np.float32)\n        nrm = raw / np.sqrt(np.maximum(cs, 1.0))\n        mfeat = [_z(raw), _rank_norm(raw), raw - raw.max(), _z(nrm), _rank_norm(nrm),\n                 (raw == raw.max()).astype(np.float32)]\n    else:\n        mfeat = [np.zeros(nc, np.float32)] * 6\n    if model_logits is not None and nc:\n        mr = _rank_norm(cf @ np.asarray(model_logits, np.float32))\n        lbest = int(np.argmax(lv)) if lvmax > 0 else -1\n        agree = float(1.0 - mr[lbest]) if lbest >= 0 else 0.0\n        abest = int(np.argmax(ap)) if apmax > 0 else -1\n        agree_a = float(1.0 - mr[abest]) if abest >= 0 else 0.0\n        xfeat = [lv * (1.0 - mr), ap * (1.0 - mr), np.full(nc, agree), np.full(nc, agree_a),\n                 np.full(nc, agree * lvmax), np.full(nc, float(np.corrcoef(lv, -mr)[0, 1]) if lv.std() > 1e-9 else 0.0)]\n    else:\n        xfeat = [np.zeros(nc, np.float32)] * 6\n    if frag is not None:\n        fr = np.asarray(frag, np.float32)\n        ffeat = [fr, _rank_norm(fr), fr - fr.max() if nc else fr, _z(fr)]\n    else:\n        ffeat = [np.zeros(nc, np.float32)] * 4\n    return np.column_stack([\n        lv, _rank_norm(lv), np.full(nc, lvmax), lv - lvmax, (lv > 0).astype(float),\n        ap, _rank_norm(ap), np.full(nc, apmax), ap - apmax,\n        a1, best_tan, top_tan, mean_tan, np.full(nc, top_sim),\n        np.full(nc, np.log(max(nc, 1))),\n        *mfeat, *ffeat, *xfeat,\n    ]).astype(np.float32)\n\n\n# ----------------------------------------------------------------------------------------------\n# MetFrag-lite\n# ----------------------------------------------------------------------------------------------\nAMU = {\'C\': 12.0, \'H\': 1.00782503207, \'N\': 14.0030740048, \'O\': 15.9949146196, \'P\': 30.97376163,\n       \'S\': 31.97207100, \'F\': 18.99840322, \'Cl\': 34.96885268, \'Br\': 78.9183371, \'I\': 126.904473,\n       \'Na\': 22.9897692809, \'K\': 38.96370668, \'Si\': 27.9769265325, \'B\': 11.0093054, \'Se\': 79.9165213}\nH = AMU[\'H\']\n\n\ndef mol_graph(smi):\n    from rdkit import Chem, RDLogger\n    RDLogger.DisableLog(\'rdApp.*\')\n    m = Chem.MolFromSmiles(smi)\n    if m is None: return None\n    n = m.GetNumAtoms()\n    w = np.zeros(n)\n    for a in m.GetAtoms():\n        w[a.GetIdx()] = AMU.get(a.GetSymbol(), 0.0) + a.GetTotalNumHs() * H\n    if (w == 0).any(): return None\n    bonds = [(b.GetBeginAtomIdx(), b.GetEndAtomIdx()) for b in m.GetBonds()]\n    return w, bonds, n\n\n\ndef _components(n, bonds, drop):\n    adj = [[] for _ in range(n)]\n    for i, (a, b) in enumerate(bonds):\n        if i in drop: continue\n        adj[a].append(b); adj[b].append(a)\n    seen = np.zeros(n, bool); comps = []\n    for s in range(n):\n        if seen[s]: continue\n        stack = [s]; seen[s] = True; cur = [s]\n        while stack:\n            u = stack.pop()\n            for v in adj[u]:\n                if not seen[v]: seen[v] = True; stack.append(v); cur.append(v)\n        comps.append(cur)\n    return comps\n\n\ndef fragment_masses(smi, max_breaks=2, max_bonds=34):\n    g = mol_graph(smi)\n    if g is None: return np.zeros(0)\n    w, bonds, n = g\n    nb = len(bonds)\n    if nb == 0 or nb > max_bonds: return np.array([w.sum()])\n    out = {w.sum()}\n    for i in range(nb):\n        for c in _components(n, bonds, {i}):\n            out.add(float(w[c].sum()))\n    if max_breaks >= 2:\n        for i in range(nb):\n            for j in range(i + 1, nb):\n                for c in _components(n, bonds, {i, j}):\n                    out.add(float(w[c].sum()))\n    return np.array(sorted(out))\n\n\ndef frag_masses_safe(smi):\n    try: return fragment_masses(smi)\n    except Exception: return np.zeros(0)\n\n\n# ADDED: charge carriers a fragment may keep, per precursor adduct (prvsiyan only used +-proton)\nCARRIERS = {\n    "[M+H]+": (PROTON,), "[M-H2O+H]+": (PROTON,), "[M-2H2O+H]+": (PROTON,),\n    "[M+NH4]+": (PROTON, NH4 - E), "[M+Na]+": (PROTON, MASS[\'Na\'] - E), "[M+K]+": (PROTON, MASS[\'K\'] - E),\n    "[M-H]-": (-PROTON,), "[M-H2O-H]-": (-PROTON,), "[M+CH2O2-H]-": (-PROTON, FORMATE - PROTON),\n    "[M+Cl]-": (-PROTON, MASS[\'Cl\'] + E),\n}\n\n\ndef explain_score_adduct(frag_mass, peak_mz, peak_int, adduct, tol=0.01, h_shifts=(-2, -1, 0, 1, 2)):\n    if len(frag_mass) == 0 or len(peak_mz) == 0: return 0.0\n    car = CARRIERS.get(adduct, (PROTON,) if "+" in adduct[-2:] else (-PROTON,))\n    ion = np.sort(np.concatenate([frag_mass + dh * H + c for dh in h_shifts for c in car]))\n    w = np.sqrt(np.asarray(peak_int, float)); tot = w.sum()\n    if tot <= 0: return 0.0\n    idx = np.searchsorted(ion, peak_mz)\n    ok = np.zeros(len(peak_mz), bool)\n    for off in (-1, 0):\n        k = np.clip(idx + off, 0, len(ion) - 1)\n        ok |= np.abs(ion[k] - peak_mz) <= tol\n    return float(w[ok].sum() / tot)\n\n\ndef explain_score(frag_mass, peak_mz, peak_int, mode=1.0, tol=0.01, h_shifts=(-2, -1, 0, 1, 2)):\n    if len(frag_mass) == 0 or len(peak_mz) == 0: return 0.0\n    ion = []\n    for dh in h_shifts:\n        ion.append(frag_mass + dh * H + (PROTON if mode > 0 else -PROTON))\n    ion = np.sort(np.concatenate(ion))\n    w = np.sqrt(np.asarray(peak_int, float)); tot = w.sum()\n    if tot <= 0: return 0.0\n    idx = np.searchsorted(ion, peak_mz)\n    ok = np.zeros(len(peak_mz), bool)\n    for off in (-1, 0):\n        k = np.clip(idx + off, 0, len(ion) - 1)\n        ok |= np.abs(ion[k] - peak_mz) <= tol\n    return float(w[ok].sum() / tot)', 'pv_fp.py': '"""Spectrum -> fingerprint model (prvsiyan FPNet), kept apart so CPU worker processes never import torch."""\nimport math\nimport numpy as np\n# ----------------------------------------------------------------------------------------------\n# spectrum -> fingerprint model\n# ----------------------------------------------------------------------------------------------\nimport torch, torch.nn as nn, torch.nn.functional as F\n\nFP_MAX_PEAKS = 128\nADDUCT_LIST = ["[M+H]+", "[M+NH4]+", "[M+Na]+", "[M+K]+", "[M-H2O+H]+", "[M-2H2O+H]+", "[M]+",\n               "[M-H]-", "[M-H2O-H]-", "[M+CH2O2-H]-", "[M+C2H4O2-H]-", "[M+Cl]-", "[M]-",\n               "[M+2H]2+", "[M-2H]-", "[2M+H]+", "[2M+Na]+", "[2M+NH4]+", "[2M-H]-", "[2M+K]+",\n               "[2M+CH2O2-H]-", "[2M+C2H4O2-H]-", "[2M+Na-2H]-", "[M+Na-2H]-", "[M-H2O]+", "<unk>"]\nADDUCT_IX = {a: i for i, a in enumerate(ADDUCT_LIST)}\nINSTR_LIST = ["timsTOF", "Orbitrap", "QTOF", "IT", "other"]\n\n\ndef instr_family(s):\n    if s is None: return 4\n    t = str(s).lower()\n    if \'timstof\' in t: return 0\n    if \'orbitrap\' in t or \'qft\' in t or \'ftms\' in t or \'hybrid ft\' in t or \'itft\' in t or \'exactive\' in t: return 1\n    if \'tof\' in t: return 2\n    if \'trap\' in t or \'qq\' in t: return 3\n    return 4\n\n\ndef prep_peaks(mz, inten, prec_mz, max_peaks=FP_MAX_PEAKS, floor=1e-3, win=50.0, per_win=8):\n    mz = np.asarray(mz, np.float64); it = np.asarray(inten, np.float64)\n    if len(mz) == 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)\n    keep = (mz <= prec_mz + 1.5)\n    mz, it = mz[keep], it[keep]\n    if len(mz) == 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)\n    mx = it.max()\n    if mx <= 0: return np.zeros(0, np.float32), np.zeros(0, np.float32)\n    keep = it >= floor * mx\n    mz, it = mz[keep], it[keep]\n    if len(mz) > max_peaks:\n        order = np.argsort(-it)\n        bucket = (mz // win).astype(np.int64)\n        cnt = {}; sel = []\n        for i in order:\n            b = bucket[i]; c = cnt.get(b, 0)\n            if c < per_win: cnt[b] = c + 1; sel.append(i)\n        sel = np.array(sel)\n        if len(sel) > max_peaks:\n            sel = sel[np.argsort(-it[sel])[:max_peaks]]\n        elif len(sel) < max_peaks:\n            rest = np.array([i for i in order if i not in set(sel.tolist())])\n            need = max_peaks - len(sel)\n            if len(rest): sel = np.concatenate([sel, rest[:need]])\n        mz, it = mz[sel], it[sel]\n    o = np.argsort(mz)\n    mz, it = mz[o], it[o]\n    v = np.sqrt(it / it.max())\n    return mz.astype(np.float32), v.astype(np.float32)\n\n\nclass SinEmb(nn.Module):\n    def __init__(self, dim, lo=-2.0, hi=3.2, power=1.0):\n        super().__init__()\n        n = dim // 2\n        wav = torch.pow(10.0, (hi - lo) * torch.pow(torch.linspace(0, 1, n), power) + lo)\n        self.register_buffer(\'inv\', (2 * math.pi) / wav)\n\n    def forward(self, x):\n        a = x.unsqueeze(-1) * self.inv\n        return torch.cat([torch.sin(a), torch.cos(a)], -1)\n\n\nclass Block(nn.Module):\n    def __init__(self, d, h, drop):\n        super().__init__(); self.h = h\n        self.n1 = nn.LayerNorm(d); self.qkv = nn.Linear(d, 3 * d); self.o = nn.Linear(d, d)\n        self.n2 = nn.LayerNorm(d)\n        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Dropout(drop), nn.Linear(4 * d, d))\n        self.drop = nn.Dropout(drop)\n\n    def forward(self, x, pad):\n        B, N, D = x.shape; y = self.n1(x)\n        q, k, v = self.qkv(y).view(B, N, 3, self.h, D // self.h).permute(2, 0, 3, 1, 4)\n        m = (~pad)[:, None, None, :]\n        a = F.scaled_dot_product_attention(q, k, v, attn_mask=m)\n        x = x + self.drop(self.o(a.transpose(1, 2).reshape(B, N, D)))\n        return x + self.drop(self.ff(self.n2(x)))\n\n\nclass FPNet(nn.Module):\n    def __init__(self, nbits, d=512, layers=6, heads=8, drop=0.1):\n        super().__init__()\n        self.d = d\n        self.mz_emb = SinEmb(d)\n        self.nl_emb = SinEmb(d)\n        self.pk = nn.Linear(2 * d + 1, d)\n        self.prec_emb = SinEmb(d)\n        self.ad = nn.Embedding(len(ADDUCT_LIST), d)\n        self.ins = nn.Embedding(len(INSTR_LIST), d)\n        self.gl = nn.Linear(d + 3, d)\n        self.blocks = nn.ModuleList([Block(d, heads, drop) for _ in range(layers)])\n        self.norm = nn.LayerNorm(d)\n        self.head = nn.Sequential(nn.Linear(2 * d, 2048), nn.GELU(), nn.Dropout(drop), nn.Linear(2048, nbits))\n\n    def forward(self, mz, it, pad, prec, ad, ins, ce, mode):\n        B, N = mz.shape\n        nl = (prec[:, None] - mz).clamp(min=0)\n        p = self.pk(torch.cat([self.mz_emb(mz), self.nl_emb(nl), it.unsqueeze(-1)], -1))\n        g = self.gl(torch.cat([self.prec_emb(prec),\n                               (ce / 100.0).unsqueeze(-1), mode.unsqueeze(-1),\n                               torch.log1p(prec).unsqueeze(-1) / 10.0], -1)) + self.ad(ad) + self.ins(ins)\n        x = torch.cat([g.unsqueeze(1), p], 1)\n        pad = torch.cat([torch.zeros(B, 1, dtype=torch.bool, device=pad.device), pad], 1)\n        for b in self.blocks: x = b(x, pad)\n        x = self.norm(x)\n        cls = x[:, 0]\n        msk = (~pad[:, 1:]).float().unsqueeze(-1)\n        mean = (x[:, 1:] * msk).sum(1) / msk.sum(1).clamp(min=1)\n        return self.head(torch.cat([cls, mean], -1))\n\n\ndef load_fp_models(paths, dev):\n    single, merged = [], []\n    nbits = None\n    for pth in paths:\n        ck = torch.load(pth, map_location=\'cpu\', weights_only=False)\n        net = FPNet(ck[\'nbits\'], d=ck[\'d\'], layers=ck[\'layers\']).to(dev).eval()\n        net.load_state_dict(ck[\'model\'])\n        (merged if \'merged\' in str(pth).replace(\'\\\\\', \'/\').split(\'/\')[-1] else single).append(net)\n        nbits = ck[\'nbits\']\n    return single, merged, nbits\n\n\ndef merge_peaks(specs):\n    """specs: list of (mz, it). All peaks collapsed into one pseudo-spectrum."""\n    specs = [(a, b) for a, b in specs if len(a)]\n    if not specs: return np.zeros(0), np.zeros(0)\n    mz = np.concatenate([np.asarray(a, float) for a, _ in specs])\n    it = np.concatenate([np.asarray(b, float) / max(float(np.asarray(b, float).max()), 1e-9) for _, b in specs])\n    o = np.argsort(mz); mz, it = mz[o], it[o]\n    keep = np.ones(len(mz), bool)\n    for j in range(1, len(mz)):\n        if mz[j] - mz[j - 1] < 0.005:\n            if it[j] >= it[j - 1]: keep[j - 1] = False\n            else: keep[j] = False\n    return mz[keep], it[keep]\n\n\n@torch.no_grad()\ndef logits_batch(P, nets, dev, prec, adduct, instr, ce, mode):\n    """P: list of (mz, it) already prepped; per-row arrays prec/adduct(str)/instr(str)/ce/mode."""\n    keep = [i for i, (a, _) in enumerate(P) if len(a)]\n    if not keep: return None\n    P = [P[i] for i in keep]\n    B = len(P); N = max(len(a) for a, _ in P)\n    mz = np.zeros((B, N), np.float32); it = np.zeros((B, N), np.float32); pad = np.ones((B, N), bool)\n    for i, (a, b) in enumerate(P):\n        mz[i, :len(a)] = a; it[i, :len(b)] = b; pad[i, :len(a)] = False\n    T = lambda x: torch.as_tensor(x, device=dev)\n    args = (T(mz), T(it), T(pad),\n            T(np.asarray(prec, np.float32)[keep]),\n            T(np.array([ADDUCT_IX.get(a, ADDUCT_IX[\'<unk>\']) for a in np.asarray(adduct, object)[keep]])),\n            T(np.array([instr_family(s) for s in np.asarray(instr, object)[keep]])),\n            T(np.asarray(ce, np.float32)[keep]),\n            T(np.asarray(mode, np.float32)[keep]))\n    return np.mean([n(*args).float().mean(0).cpu().numpy() for n in nets], axis=0)\n\n\ndef molecule_logits(models, specs, prec, adduct, instr, ce, mode):\n    """prvsiyan model_logits(): mean of (single-model per-spectrum mean) and (merged-model on merged list).\n    specs: list of (mz, it); prec/adduct/instr/ce/mode: per-spectrum."""\n    single, merged, dev = models\n    out = []\n    if single:\n        P = [prep_peaks(a, b, p) for (a, b), p in zip(specs, prec)]\n        za = logits_batch(P, single, dev, prec, adduct, instr, ce, mode)\n        if za is not None: out.append(za)\n    if merged:\n        mz, it = merge_peaks(specs)\n        pm = float(np.median(prec))\n        P = [prep_peaks(mz, it, pm)]\n        zb = logits_batch(P, merged, dev, [pm], [adduct[0]], [instr[0]], [25.0], [float(np.mean(mode))])\n        if zb is not None: out.append(zb)\n    return np.mean(out, axis=0) if out else None', 'casmi_engine.py': '"""End-to-end inference (Kaggle notebook body). Mirrors e02_channels.py + e05_rank.py feature blocks exactly,\nbut reads train.parquet / test.parquet directly. Runs locally too (set CASMI_LOCAL=1).\n\nStages: library -> candidate pool -> index -> per-molecule channels -> MetFrag-lite -> features -> ranker -> submission\n"""\nimport os, sys, glob, time, pickle, math\nos.environ.setdefault("OPENBLAS_NUM_THREADS", "1")\nimport numpy as np, pandas as pd, pyarrow.parquet as pq, pyarrow as pa\nimport pv\nfrom pv import CFG\nimport re as _re\n_MONO = dict(C=12.0, H=1.00782503207, N=14.0030740048, O=15.9949146196, S=31.97207100, P=30.97376163,\n             Cl=34.96885268, Br=78.9183371, F=18.99840322, I=126.904473, Si=27.9769265325, Se=79.9165213,\n             Na=22.9897692809, K=38.96370668, B=11.0093054, As=74.9215965, Fe=55.9349375, D=2.0141017778)\n_TOK = _re.compile(r"([A-Z][a-z]?)(\\d*)")\n\n\ndef formula_mass(f):\n    """ADDED: neutral monoisotopic mass by arithmetic on molecular_formula (prvsiyan fix; immune to\n    riken\'s 0.005 Da precursor precision and gnps adduct mislabels). NaN if an element is untabulated."""\n    if not isinstance(f, str) or not f: return np.nan\n    m = 0.0\n    for el, n in _TOK.findall(f):\n        if el not in _MONO: return np.nan\n        m += _MONO[el] * (int(n) if n else 1)\n    return m\n\nPOOL = None\n\n\nclass RANK:\n    W_A = (0.35, 0.55)      # weight on class-1 simulation rows; two priors averaged\n    SEEDS = (0, 1)\n    GBM = dict(max_depth=6, max_iter=500, learning_rate=0.03, min_samples_leaf=80, l2_regularization=1.0)\n\nT0 = time.time()\nLOCAL = os.environ.get("CASMI_LOCAL") == "1"\nROOTS = [r"C:\\Users\\HW-LEE\\Desktop\\CASMI"] if LOCAL else ["/kaggle/input"]\n\n\ndef find(name):\n    for root in ROOTS:\n        hits = glob.glob(os.path.join(root, "**", name), recursive=True)\n        if hits: return sorted(hits, key=len)[0]\n    raise FileNotFoundError(name)\n\n\ndef log(msg):\n    print(f"[{time.time()-T0:6.0f}s] {msg}", flush=True)\n\n\n# ---------------------------------------------------------------------------------------------\n# library\n# ---------------------------------------------------------------------------------------------\ndef load_library(path):\n    """Row group by row group, peaks straight to float32: peak RAM stays near the final ~3.5 GB."""\n    f = pq.ParquetFile(path)\n    n_rows = f.metadata.num_rows\n    npk = 0\n    for i in range(f.num_row_groups):\n        c = f.read_row_group(i, columns=["ms2_mzs"]).column(0).combine_chunks()\n        npk += len(c.values)\n    off = np.zeros(n_rows + 1, np.int64)\n    allmz = np.empty(npk, np.float32); allin = np.empty(npk, np.float32)\n    prec, add, pol, ik, smi = [], [], [], [], []\n    fo, tims = [], []\n    r0 = p0 = 0\n    for i in range(f.num_row_groups):\n        t = f.read_row_group(i, columns=["inchikey14", "normalized_smiles", "adduct", "precursor_mz", "ionization_mode",\n                                         "ms2_mzs", "ms2_normalized_intensities", "molecular_formula", "instrument_type"])\n        mzc = t.column("ms2_mzs").combine_chunks(); itc = t.column("ms2_normalized_intensities").combine_chunks()\n        o = mzc.offsets.to_numpy().astype(np.int64)\n        n = len(o) - 1; k = int(o[-1] - o[0])\n        allmz[p0:p0 + k] = mzc.values.to_numpy(zero_copy_only=False)[o[0]:o[-1]]\n        allin[p0:p0 + k] = itc.values.to_numpy(zero_copy_only=False)[o[0]:o[-1]]\n        off[r0 + 1:r0 + n + 1] = p0 + (o[1:] - o[0])\n        prec.append(t.column("precursor_mz").to_numpy(zero_copy_only=False).astype(np.float64))\n        add += t.column("adduct").cast(pa.string()).to_pylist()\n        pol.append(np.array(t.column("ionization_mode").cast(pa.string()).to_pylist(), dtype=object) == "positive")\n        ik += t.column("inchikey14").cast(pa.string()).to_pylist()\n        smi += t.column("normalized_smiles").cast(pa.string()).to_pylist()\n        fo += t.column("molecular_formula").cast(pa.string()).to_pylist()\n        tims.append(np.array([isinstance(x, str) and x.lower().replace("-", "").replace(" ", "") == "timstof"\n                              for x in t.column("instrument_type").cast(pa.string()).to_pylist()]))\n        r0 += n; p0 += k\n        del t, mzc, itc\n    prec = np.concatenate(prec); add = np.asarray(add, dtype=object)\n    pol = np.where(np.concatenate(pol), 1, -1).astype(np.int8)\n    nmp = pv.neutral_mass(prec, add)\n    _fm = {x: formula_mass(x) for x in set(v for v in fo if isinstance(v, str))}\n    nmf = np.array([_fm.get(x, np.nan) if isinstance(x, str) else np.nan for x in fo])\n    nm = np.where(np.isfinite(nmf), nmf, nmp)\n    tims = np.concatenate(tims)\n    log(f"library mass index: formula {np.isfinite(nmf).mean():.1%}, timsTOF rows {tims.mean():.1%}")\n    del fo\n    return dict(off=off, mz=allmz, it=allin, prec=prec, nm=nm, pol=pol, tims=tims,\n                ik=np.asarray(ik, dtype=object), smi=np.asarray(smi, dtype=object))\n\n\n# ---------------------------------------------------------------------------------------------\n# candidate pool = COCONUT (prvsiyan fingerprints) U training structures (fingerprinted here)\n# ---------------------------------------------------------------------------------------------\n_g = {}\n\n\ndef _fp_init():\n    from rdkit import RDLogger\n    from rdkit.Chem import rdFingerprintGenerator\n    RDLogger.DisableLog("rdApp.*")\n    _g["bits"] = np.load(find("fp_bits.npy"))\n    _g["m2"] = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=4096)\n    _g["m3"] = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=4096)\n    _g["rk"] = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048, maxPath=6)\n\n\ndef fp_and_mass(smi):\n    from rdkit import Chem\n    from rdkit.Chem import MACCSkeys\n    from rdkit.Chem.Descriptors import ExactMolWt\n    if not _g: _fp_init()\n    m = Chem.MolFromSmiles(smi)\n    if m is None: return None\n    try:\n        fp = np.concatenate([_g["m2"].GetFingerprintAsNumPy(m).astype(np.uint8),\n                             _g["m3"].GetFingerprintAsNumPy(m).astype(np.uint8),\n                             _g["rk"].GetFingerprintAsNumPy(m).astype(np.uint8),\n                             np.array(MACCSkeys.GenMACCSKeys(m), dtype=np.uint8)])[_g["bits"]]\n        return np.packbits(fp), float(ExactMolWt(m))\n    except Exception:\n        return None\n\n\ndef canon_key(smi):\n    from rdkit import Chem, RDLogger\n    from rdkit.Chem.MolStandardize import rdMolStandardize\n    RDLogger.DisableLog("rdApp.*")\n    if "te" not in _g: _g["te"] = rdMolStandardize.TautomerEnumerator()\n    try:\n        m = Chem.MolFromSmiles(smi)\n        return Chem.MolToInchiKey(_g["te"].Canonicalize(m))[:14] if m is not None else None\n    except Exception:\n        return None\n\n\ndef build_pool(L, workers):\n    from multiprocessing import Pool as MPool\n    d = os.path.dirname(find("coco_fp.npy"))\n    cm = pickle.load(open(os.path.join(d, "coco_meta.pkl"), "rb"))\n    co_fp = np.load(os.path.join(d, "coco_fp.npy")); co_mass = np.load(os.path.join(d, "coco_mass.npy"))\n    co_keys = np.asarray(cm["keys"], dtype=object); co_smi = np.asarray(cm["smiles"], dtype=object)\n    # ADDED: ChEBI + LIPID MAPS (prvsiyan/chebi-lipidmaps-casmi26), same 6,930-bit packed fingerprints\n    bd = os.path.dirname(find("bio_fp.npy")); bm = pickle.load(open(os.path.join(bd, "bio_meta.pkl"), "rb"))\n    bkeys = np.asarray(bm["keys"], dtype=object); bnew = ~pd.Series(bkeys).isin(set(co_keys)).values\n    co_fp = np.vstack([co_fp, np.load(os.path.join(bd, "bio_fp.npy"))[bnew]])\n    co_mass = np.concatenate([co_mass, np.load(os.path.join(bd, "bio_mass.npy"))[bnew]])\n    co_keys = np.concatenate([co_keys, bkeys[bnew]])\n    co_smi = np.concatenate([co_smi, np.asarray(bm["smiles"], dtype=object)[bnew]])\n    log(f"+ ChEBI/LIPID MAPS {int(bnew.sum()):,} new structures")\n    tr = pd.DataFrame({"ik": L["ik"], "smi": L["smi"]}).dropna().drop_duplicates("ik")\n    tr = tr[~tr.ik.isin(set(co_keys))]\n    log(f"COCONUT {len(co_keys):,}; training structures to fingerprint {len(tr):,}")\n    with MPool(workers) as mp:\n        res = mp.map(fp_and_mass, list(tr.smi), chunksize=500)\n    ok = np.array([r is not None for r in res])\n    tr_fp = np.stack([r[0] for r in res if r is not None]); tr_mass = np.array([r[1] for r in res if r is not None])\n    fp = np.vstack([co_fp, tr_fp]); mass = np.concatenate([co_mass, tr_mass])\n    keys = np.concatenate([co_keys, tr.ik.values[ok]]); smis = np.concatenate([co_smi, tr.smi.values[ok]])\n    good = np.isfinite(mass)\n    o = np.argsort(np.where(good, mass, 1e18), kind="mergesort")[: good.sum()]\n    P = dict(fp=fp[o], mass=mass[o], keys=keys[o], smiles=smis[o])\n    P["k2i"] = pd.Series(np.arange(len(o)), index=P["keys"]); P["k2i"] = P["k2i"][~P["k2i"].index.duplicated()]\n    log(f"pool {len(o):,} structures")\n    return P\n\n\ndef pool_fps(idx):\n    return np.unpackbits(np.asarray(POOL["fp"][np.asarray(idx, np.int64)]), axis=1)[:, :6930]\n\n\ndef pool_window(t, ppm):\n    a = np.searchsorted(POOL["mass"], t * (1 - ppm / 1e6), "left")\n    b = np.searchsorted(POOL["mass"], t * (1 + ppm / 1e6), "right")\n    return np.arange(a, b)\n\n\n# ---------------------------------------------------------------------------------------------\n# index: neutral-mass sorted library rows + cleaned representatives (all / positive / negative)\n# ---------------------------------------------------------------------------------------------\ndef _reps(L, rows):\n    npk = np.diff(L["off"])[rows]\n    df = pd.DataFrame({"row": rows, "p": L["row_p"][rows], "npk": npk, "tims": L["tims"][rows]})\n    # ADDED: prefer the test instrument (timsTOF) representative, then the richest spectrum\n    df = df.sort_values(["p", "tims", "npk", "row"], ascending=[True, False, False, True]).drop_duplicates("p")\n    rep = df.row.values; rep_nm = L["nm"][rep]\n    o = np.argsort(rep_nm, kind="mergesort"); rep = rep[o]\n    roff, rmz, rit = pv.clean_store(rep, L["off"], L["mz"], L["it"], CFG.INT_FLOOR, CFG.MAX_PEAKS, CFG.INT_POWER, CFG.ENT_WEIGHT)\n    return dict(rep=rep, rep_p=L["row_p"][rep], rep_nm=L["nm"][rep], roff=roff, rmz=rmz, rit=rit)\n\n\ndef build_index(L):\n    ok = np.isfinite(L["nm"]) & (L["row_p"] >= 0)\n    rows = np.where(ok)[0]\n    o = np.argsort(L["nm"][rows], kind="mergesort")\n    I = dict(lib_rows=rows[o], lib_nm=L["nm"][rows[o]])\n    I["all"] = _reps(L, rows)\n    I["pos"] = _reps(L, rows[L["pol"][rows] == 1])\n    I["neg"] = _reps(L, rows[L["pol"][rows] == -1])\n    log(f"index: {len(rows):,} library rows; representatives {len(I[\'all\'][\'rep\']):,}")\n    return I\n\n\n# ---------------------------------------------------------------------------------------------\n# per-molecule channels (identical to e02_channels.molecule_channels)\n# ---------------------------------------------------------------------------------------------\ndef clean(mz, it):\n    return pv._clean(np.asarray(mz, np.float32), np.asarray(it, np.float32),\n                     CFG.INT_FLOOR, CFG.MAX_PEAKS, CFG.INT_POWER, CFG.ENT_WEIGHT)\n\n\ndef analog_search(R, Q, target):\n    lo = np.searchsorted(R["rep_nm"], target - CFG.ANALOG_WIN, "left")\n    hi = np.searchsorted(R["rep_nm"], target + CFG.ANALOG_WIN, "right")\n    asim = np.zeros(hi - lo, np.float32)\n    if hi > lo and Q:\n        shift = (target - R["rep_nm"][lo:hi]).astype(np.float32)\n        for qm, qp in Q:\n            asim = np.maximum(asim, pv.search_shift_pre(qm, qp, lo, hi, R["roff"], R["rmz"], R["rit"], CFG.MZ_TOL, shift))\n    return R["rep_p"][lo:hi], asim\n\n\ndef top_analogs(p, s):\n    if len(p) == 0: return []\n    df = pd.DataFrame({"p": p, "s": s}).groupby("p", sort=False).s.max().sort_values(ascending=False, kind="mergesort")\n    return list(zip(df.index[:CFG.N_ANALOG].astype(int), df.values[:CFG.N_ANALOG].astype(float)))\n\n\ndef molecule_channels(L, I, specs, precs, pols, target):\n    Q = [clean(a, b) for a, b in specs]\n    keep = [i for i, (a, _) in enumerate(Q) if len(a)]\n    Q = [Q[i] for i in keep]; precs = np.asarray(precs)[keep]; pols = np.asarray(pols)[keep]\n    tol = target * CFG.PPM_WIN / 1e6\n    lo = np.searchsorted(I["lib_nm"], target - tol, "left"); hi = np.searchsorted(I["lib_nm"], target + tol, "right")\n    crow = I["lib_rows"][lo:hi]\n    u, inv = np.unique(L["row_p"][crow], return_inverse=True)\n    M = np.zeros((max(len(Q), 1), len(u)), np.float32); MS = np.zeros_like(M)\n    for j, (qm, qp) in enumerate(Q):\n        if len(crow) == 0: break\n        s = pv.search(qm, qp, crow, L["off"], L["mz"], L["it"], CFG.MZ_TOL, CFG.INT_FLOOR, CFG.MAX_PEAKS,\n                      CFG.INT_POWER, CFG.ENT_WEIGHT)\n        np.maximum.at(M[j], inv, s)\n        sh = (precs[j] - L["prec"][crow]).astype(np.float32)\n        s2 = pv.search_shift_rows(qm, qp, crow, L["off"], L["mz"], L["it"], CFG.MZ_TOL, CFG.INT_FLOOR, CFG.MAX_PEAKS,\n                                  CFG.INT_POWER, CFG.ENT_WEIGHT, sh)\n        np.maximum.at(MS[j], inv, s2)\n    cnt = np.bincount(inv, minlength=len(u)).astype(np.float32)\n    z = np.zeros(0, np.float32)\n    lib = (u, M.max(0) if len(u) else z, M.mean(0) if len(u) else z, MS.max(0) if len(u) else z, cnt)\n    p, s = analog_search(I["all"], Q, target)\n    ana = top_analogs(p, s)\n    ps, ss = [], []\n    for sign, name in ((1, "pos"), (-1, "neg")):\n        QQ = [q for q, pl in zip(Q, pols) if pl == sign]\n        if QQ:\n            p, s = analog_search(I[name], QQ, target); ps.append(p); ss.append(s)\n    anapol = top_analogs(np.concatenate(ps), np.concatenate(ss)) if ps else []\n    return lib, ana, anapol\n\n\ndef parse_ce(v):\n    try:\n        a = np.abs(np.atleast_1d(np.asarray(v, float)))\n        return float(a.mean()) if len(a) else 25.0\n    except Exception:\n        return 25.0\n\n\n\n\n# ---------------------------------------------------------------------------------------------\n# ranking features: prvsiyan\'s 31 + our blocks (identical to e05_rank.py BLOCKS)\n# ---------------------------------------------------------------------------------------------\ndef _rel(v):\n    v = np.asarray(v, np.float32)\n    return [v, pv._rank_norm(v).astype(np.float32), v - (v.max() if len(v) else 0.0)]\n\n\ndef _analog_feats(cfp, ana):\n    nc = len(cfp)\n    if not ana: return np.zeros((nc, 5), np.float32)\n    afp = pool_fps([p for p, _ in ana]).astype(np.float32); cf = cfp.astype(np.float32)\n    inter = cf @ afp.T\n    tan = inter / (cf.sum(1)[:, None] + afp.sum(1)[None, :] - inter + 1e-9)\n    w = np.clip(np.array([s for _, s in ana], np.float32), 0, None)\n    ap = (tan * (w ** 3)[None, :]).max(1)\n    ap6 = (tan * (w ** 6)[None, :]).max(1)\n    return np.column_stack(_rel(ap) + [ap6, np.full(nc, w[0])]).astype(np.float32)\n\n\ndef blk_base(r, cfp):\n    ana = r["ana"]\n    afp = pool_fps([p for p, _ in ana]) if ana else None\n    sims = np.array([s for _, s in ana], np.float32)\n    return pv.rank_features(cfp, r["lib"], afp, sims, r["zlog"], r["frag"])\n\n\ndef blk_mass(r, cfp):\n    ppm = (POOL["mass"][r["cand"]] - r["target"]) / r["target"] * 1e6\n    return np.column_stack([ppm, np.abs(ppm), np.abs(ppm + 1.5)]).astype(np.float32)\n\n\ndef blk_libx(r, cfp):\n    cnt = np.log1p(r["lib_cnt"])\n    return np.column_stack(_rel(r["lib_mean"]) + _rel(r["libsh"]) + [cnt, cnt - cnt.max()]).astype(np.float32)\n\n\ndef blk_anapol(r, cfp):\n    return _analog_feats(cfp, r["anapol"])\n\n\ndef blk_fragx(r, cfp):\n    return np.column_stack(_rel(r["frag_ad"]) + _rel(r["frag_mean"]) + [r["frag_pol"]]).astype(np.float32)\n\n\nBLOCKS = {"base": blk_base, "mass": blk_mass, "libx": blk_libx, "anapol": blk_anapol, "fragx": blk_fragx}\n\n\n# ---------------------------------------------------------------------------------------------\n# run\n# ---------------------------------------------------------------------------------------------\ndef compute_channels(L, I, models, te):\n    mols = list(te.groupby("molecule_id", sort=True))\n    recs = []\n    for gi, (mid, sub) in enumerate(mols):\n        specs = []\n        for r in sub.itertuples():\n            a_ = np.asarray(r.ms2_mzs, np.float32); b_ = np.asarray(r.ms2_normalized_intensities, np.float32)\n            k_ = a_ <= float(r.precursor_mz) + 2.0\n            a_, b_ = a_[k_], b_[k_]\n            specs.append((a_, b_ / b_.max() if len(b_) and b_.max() > 0 else b_))\n        nms = pv.neutral_mass(sub.precursor_mz.values.astype(np.float64), sub.adduct.astype(str).values)\n        nms = nms[np.isfinite(nms)]\n        rec = dict(mid=mid, cand=np.zeros(0, np.int64))\n        if len(nms):\n            target = float(np.median(nms))\n            modes = np.where(sub.ionization_mode.astype(str).values == "positive", 1.0, -1.0)\n            lib, ana, anapol = molecule_channels(L, I, specs, sub.precursor_mz.values.astype(np.float64), modes, target)\n            cand = pool_window(target, CFG.PPM_WIN)\n            if len(cand) == 0: cand = pool_window(target, CFG.PPM_FALLBACK)\n            ces = np.array([parse_ce(v) for v in sub.collision_energy_ev.values], np.float32)\n            import pv_fp\n            zlog = pv_fp.molecule_logits(models, specs, sub.precursor_mz.values.astype(np.float32),\n                                         list(sub.adduct.astype(str).values), list(sub.instrument_type.astype(str).values),\n                                         ces, modes)\n            u, lmax, lmean, lsh, lcnt = lib\n            pos = {int(x): i for i, x in enumerate(u)}\n            ci = np.array([pos.get(int(c), -1) for c in cand], np.int64); has = ci >= 0\n            rec.update(target=target, cand=cand, zlog=None if zlog is None else zlog.astype(np.float32), specs=specs,\n                       mode=float(np.mean(modes)), modes=modes, adducts=list(sub.adduct.astype(str).values),\n                       ana=ana, anapol=anapol)\n            for nm_, arr in (("lib", lmax), ("lib_mean", lmean), ("libsh", lsh), ("lib_cnt", lcnt)):\n                v = np.zeros(len(cand), np.float32); v[has] = arr[ci[has]]; rec[nm_] = v\n        recs.append(rec)\n        if gi % 50 == 0: log(f"  channels {gi}/{len(mols)}")\n    return mols, recs\n\n\ndef compute_frag(recs, workers):\n    from multiprocessing import Pool as MPool\n    uc = np.unique(np.concatenate([r["cand"] for r in recs]))\n    log(f"MetFrag-lite on {len(uc):,} candidates")\n    with MPool(workers) as mp:\n        fr = mp.map(pv.frag_masses_safe, list(POOL["smiles"][uc]), chunksize=16)\n    fmap = dict(zip(uc.tolist(), fr))\n    for r in recs:\n        if not len(r["cand"]): continue\n        peaks = []\n        for a_, b_ in r.pop("specs"):\n            m2, i2 = pv._clean(a_, b_, CFG.INT_FLOOR, CFG.MAX_PEAKS, 1.0, False)\n            peaks.append((np.asarray(m2, float), np.asarray(i2, float)))\n        F1 = np.zeros((len(r["cand"]), max(len(peaks), 1)), np.float32); F2 = np.zeros_like(F1); F3 = np.zeros_like(F1)\n        for j, c in enumerate(r["cand"]):\n            fm = fmap[int(c)]\n            for k, (x, y) in enumerate(peaks):\n                F1[j, k] = pv.explain_score(fm, x, y, mode=r["mode"], tol=CFG.MZ_TOL)\n                F2[j, k] = pv.explain_score_adduct(fm, x, y, r["adducts"][k], tol=CFG.MZ_TOL)\n                F3[j, k] = pv.explain_score(fm, x, y, mode=r["modes"][k], tol=CFG.MZ_TOL)\n        r["frag"] = F1.max(1); r["frag_ad"] = F2.max(1); r["frag_mean"] = F3.mean(1); r["frag_pol"] = F3.max(1)\n\n\ndef _hgb(X, Y, W, seed, gbm):\n    from sklearn.ensemble import HistGradientBoostingClassifier\n    m = HistGradientBoostingClassifier(random_state=seed, **gbm); m.fit(X, Y, sample_weight=W)\n    return m\n\n\ndef fit_rankers(rank_train_path):\n    """Our ranker: simulation rows from 7 query sets (S = 0 class-1 sim, 1 = class-2 sim)."""\n    z = np.load(rank_train_path, allow_pickle=True)\n    X, Y, S = z["X"], z["Y"], z["S"]\n    rankers = [_hgb(X, Y, np.where(S == 0, wA, 1.0 - wA), sd, RANK.GBM) for wA in RANK.W_A for sd in RANK.SEEDS]\n    log(f"our ranker: {len(rankers)} GBMs on {X.shape[0]:,} rows x {X.shape[1]} features")\n    return rankers, [str(b) for b in z["blocks"]]\n\n\ndef fit_pv_rankers(pv_rank_path, priors=(0.30, 0.60), seeds=(0, 1, 2, 3)):\n    """prvsiyan\'s ranker, exactly as in his notebook: shipped rank_train.npz (M = 0 class-1 sim)."""\n    z = np.load(pv_rank_path)\n    rankers = [_hgb(z["X"], z["Y"], np.where(z["M"] == 0, w1, 1.0 - w1), sd, CFG.GBM) for w1 in priors for sd in seeds]\n    log(f"prvsiyan ranker: {len(rankers)} GBMs on {z[\'X\'].shape[0]:,} rows x {z[\'X\'].shape[1]} features")\n    return rankers, z["X"].shape[1]\n\n\ndef score_molecules(recs, rankers, blocks, use_fp):\n    out = {}\n    for r in recs:\n        if not len(r["cand"]): continue\n        rr = r if use_fp else dict(r, zlog=None)\n        cfp = pool_fps(r["cand"])\n        Xm = np.hstack([BLOCKS[b](rr, cfp) for b in blocks]).astype(np.float32)\n        out[r["mid"]] = np.mean([m.predict_proba(Xm)[:, 1] for m in rankers], axis=0)\n    return out\n\n\ndef blend_scores(a, b, wa):\n    """Within-molecule rank blend (both inputs are per-candidate probabilities)."""\n    out = {}\n    for mid in a:\n        ra = 1.0 - pv._rank_norm(a[mid]); rb = 1.0 - pv._rank_norm(b[mid])\n        out[mid] = wa * ra + (1.0 - wa) * rb + 1e-6 * a[mid]\n    return out\n\n\ndef make_submissions(mols, recs, score_sets, sample_path, workers, topn=25):\n    """score_sets: {name: {mid: scores}} -> {name: submission DataFrame}; one canonicalisation pass for all."""\n    from multiprocessing import Pool as MPool\n    cand = {r["mid"]: r["cand"] for r in recs if len(r["cand"])}\n    ordered = {n: {mid: cand[mid][np.argsort(-sc, kind="mergesort")][:topn + 15] for mid, sc in S.items()}\n               for n, S in score_sets.items()}\n    allc = [v for o in ordered.values() for v in o.values()]\n    short = np.unique(np.concatenate(allc)) if allc else np.zeros(0, np.int64)\n    with MPool(workers) as mp:\n        ck = mp.map(canon_key, list(POOL["smiles"][short]), chunksize=64)\n    ckey = dict(zip(short.tolist(), ck))\n    samp = pd.read_csv(sample_path)\n    subs = {}\n    for n, o in ordered.items():\n        rows = []\n        for mid, _ in mols:\n            out, seen = [], set()\n            for c in o.get(mid, []):\n                k = ckey.get(int(c)) or POOL["keys"][c]\n                if k in seen: continue\n                seen.add(k); out.append(POOL["smiles"][c])\n                if len(out) == topn: break\n            while len(out) < topn:\n                out.append("CCO")\n            rows.append((mid, ";".join(out[:topn])))\n        sub = samp[["molecule_id"]].merge(pd.DataFrame(rows, columns=["molecule_id", "smiles"]), on="molecule_id", how="left")\n        sub["smiles"] = sub["smiles"].fillna("CCO")\n        assert len(sub) == len(samp) and sub.molecule_id.duplicated().sum() == 0 and sub.smiles.isnull().sum() == 0\n        assert (sub.smiles.str.split(";").map(len) <= topn).all()\n        subs[n] = sub\n    return subs\n\n\ndef main(test_path, train_path, sample_path, our_rank_path, pv_rank_path, fp_model_paths, workers=4, limit=0, w_pv=0.88):\n    global POOL\n    import torch\n    import pv_fp\n    L = load_library(train_path); log(f"library {len(L[\'prec\']):,} spectra")\n    POOL = build_pool(L, workers)\n    L["row_p"] = POOL["k2i"].reindex(L["ik"]).fillna(-1).values.astype(np.int64)\n    del L["ik"], L["smi"]\n    I = build_index(L)\n    dev = "cuda" if torch.cuda.is_available() else "cpu"\n    single, merged, _ = pv_fp.load_fp_models(fp_model_paths, dev)\n    log(f"fp models: {len(single)} single + {len(merged)} merged on {dev}")\n    te = pq.read_table(test_path).to_pandas()\n    if limit:\n        keep = sorted(te.molecule_id.unique())[:limit]; te = te[te.molecule_id.isin(keep)]\n    log(f"test: {len(te):,} spectra / {te.molecule_id.nunique():,} molecules")\n    mols, recs = compute_channels(L, I, (single, merged, dev), te)\n    del L, I\n    compute_frag(recs, workers)\n    ours, blocks = fit_rankers(our_rank_path)\n    s_ours = score_molecules(recs, ours, blocks, use_fp=False)\n    del ours\n    pvr, nfeat = fit_pv_rankers(pv_rank_path)\n    s_pv = score_molecules(recs, pvr, ["base"], use_fp=True)\n    del pvr\n    subs = make_submissions(mols, recs, {"blend": blend_scores(s_pv, s_ours, w_pv), "pv": s_pv, "ours": s_ours},\n                            sample_path, workers)\n    log("submissions ready")\n    return subs, recs, {"pv": s_pv, "ours": s_ours}\n'}
for _n, _src in ENG_FILES.items():
    open(os.path.join(ENG_DIR, _n), 'w').write(_src)
open(os.path.join(ENG_DIR, 'eng_runner.py'), 'w').write('"""Our engine (two-ranker, BIO + AFIX) -> {molecule_id: {smiles, keys, scores, lib_max}} (blend order).\n\nE3 version of the E1 runner.  The first 40 entries are exactly E1\'s list (first 80 candidates in blend order,\nde-duplicated on the metric key, at most 40).  When those 40 are full the scan goes on up to 60 entries, so the\nforward-model stage can form formula groups over a top 60.  scores = blend score of each kept candidate,\nlib_max = best library similarity of any candidate of the molecule (used only to order the forward-model work).\n"""\nimport os, sys, json, time\nimport numpy as np\nif __name__ == \'__main__\':\n    here = os.path.dirname(os.path.abspath(__file__)); sys.path.insert(0, here)\n    test, train, sample, out = sys.argv[1:5]\n    limit = int(sys.argv[5]) if len(sys.argv) > 5 else 0\n    keep = int(sys.argv[6]) if len(sys.argv) > 6 else 60\n    import casmi_engine as E\n    _orig_find = E.find\n    def _find(n):                                  # v4g ships its own fp_bits.npy (10,226 bits): take prvsiyan\'s 6,930\n        if n == \'fp_bits.npy\':\n            return os.path.join(os.path.dirname(_orig_find(\'coco_fp.npy\')), \'fp_bits.npy\')\n        return _orig_find(n)\n    E.find = _find\n    E.RANK.W_A = (0.35, 0.55); E.RANK.SEEDS = (0, 1); E.CFG.N_ANALOG = 200\n    import glob\n    fp_models = sorted(p for p in glob.glob(\'/kaggle/input/**/fp_*.pt\', recursive=True) if \'casmi26-fp-models-v2\' in p)\n    print(\'engine fp models\', fp_models, flush=True)\n    T0 = time.time()\n    subs, recs, S = E.main(test, train, sample, E.find(\'sim_rank_rows_nofp.npz\'), E.find(\'rank_train.npz\'), fp_models,\n                           workers=os.cpu_count(), limit=limit, w_pv=0.88)\n    bl = E.blend_scores(S[\'pv\'], S[\'ours\'], 0.88)\n    res = {}\n    for r in recs:\n        mid = r[\'mid\']\n        if mid not in bl:\n            continue\n        order = np.argsort(-bl[mid], kind=\'mergesort\')[:max(80, 3 * keep)]\n        smis, keys, scs, libs, seen = [], [], [], [], set()\n        lib = r.get(\'lib\')\n        for j, c in enumerate(r[\'cand\'][order]):\n            if j >= 80 and len(smis) < 40:         # E1 stops after 80 candidates\n                break\n            s = E.POOL[\'smiles\'][c]; k = E.canon_key(s)\n            if k is None or k in seen:\n                continue\n            seen.add(k); smis.append(s); keys.append(k); scs.append(float(bl[mid][order[j]]))\n            libs.append(float(lib[order[j]]) if lib is not None and len(lib) else 0.0)\n            if len(smis) >= max(40, keep):\n                break\n        res[str(mid)] = dict(smiles=smis, keys=keys, scores=scs, lib=libs,\n                             lib_max=float(np.max(lib)) if lib is not None and len(lib) else 0.0)\n    json.dump(res, open(out, \'w\'))\n    try:   # E7: the engine\'s library analogs (top_analogs output) for the Class-3 channel; eng_lists.json is unchanged\n        ana = {}\n        for r in recs:\n            if r.get(\'ana\') is None or \'target\' not in r:\n                continue\n            ana[str(r[\'mid\'])] = dict(target=float(r[\'target\']), adducts=[str(x) for x in r.get(\'adducts\', [])],\n                                      ana=[[str(E.POOL[\'smiles\'][p]), str(E.POOL[\'keys\'][p]), float(s), float(E.POOL[\'mass\'][p])]\n                                           for p, s in list(r[\'ana\'])[:100]])\n        json.dump(ana, open(os.path.join(os.path.dirname(os.path.abspath(out)), \'eng_ana.json\'), \'w\'))\n        print(\'engine analogs exported\', len(ana), flush=True)\n    except Exception as e:\n        print(\'ENGINE ANALOG EXPORT FAILED (E7 C3 channel will be skipped):\', repr(e), flush=True)\n    print(\'engine lists\', len(res), f\'{time.time()-T0:.0f}s\', flush=True)\n')
ENG = {}
try:
    _t0 = time.time()
    rr = subprocess.run([sys.executable, os.path.join(ENG_DIR, 'eng_runner.py'), os.path.join(COMP, 'test.parquet'),
                         os.path.join(COMP, 'train.parquet'), os.path.join(COMP, 'sample_submission.csv'),
                         '/kaggle/working/eng_lists.json', str(SMOKE_N), str(FM_TOPN)], capture_output=True, text=True, timeout=4 * 3600, cwd=ENG_DIR)
    print(rr.stdout[-3000:]); print(rr.stderr[-3000:])
    ENG = json.load(open('/kaggle/working/eng_lists.json'))
    print('engine lists', len(ENG), f'{time.time()-_t0:.0f}s')
except Exception as e:
    print('ENGINE FAILED -> v4g lists kept:', repr(e))


In [ ]:
# Forward-model re-scoring (GLACIER + ICEBERG, ms-pred MIT, MassSpecGym weights) of same-formula candidates.
# Runs in subprocesses with their own RDKit 2025.3.6 site directory; this process keeps the metric's RDKit.
# Any failure leaves ENG exactly as the engine wrote it.
import traceback, copy
E3_DIR = '/kaggle/working/e3code'; os.makedirs(E3_DIR, exist_ok=True)
E3_FILES = {'fm_rerank.py': '"""Same-formula re-ordering of a ranked candidate list with forward-model scores (ours, MIT).\n\nDefinition (per molecule; ``L`` = the engine\'s de-duplicated list, best first):\n\n1. Only the first ``topn`` entries take part.  They are split into groups of\n   identical molecular formula.  A candidate whose formula is unknown (None)\n   is a group of its own.\n2. A forward model "covers" a group when it scored at least two of its members\n   and those scores are not all equal.  A group covered by no model is left\n   exactly as it is.\n3. In a covered group every member gets\n       fused = z(ranker) + sum_m lam_m * z_m\n   * z(ranker): z-score of the ranker score over ALL members of the group\n     (sample standard deviation, ddof = 1; 0 for every member when the spread\n     is 0).  If any ranker score of the group is missing or not finite, the\n     group\'s ranker scores are replaced by minus the list position.\n   * z_m: z-score of model m over the members it scored (ddof = 1).  A member\n     the model did not score gets z_m = 0, i.e. it is placed at the mean of the\n     scored members for that model; a model that does not cover the group\n     contributes 0 for every member.  A candidate without any forward score is\n     therefore ordered by its ranker z-score alone.\n4. The group\'s members are sorted by fused value (descending, rounded to 1e-9;\n   exact ties keep their current relative order) and written back into the\n   positions the group already occupied.  Positions of other formulas, and\n   everything after ``topn``, never change.\n\nThe function returns a permutation: ``perm[new_position] = old_position``.\n"""\nimport math\n\nGRID = 1e-9\n\n\ndef _ok(v):\n    return v is not None and isinstance(v, (int, float)) and math.isfinite(v)\n\n\ndef zscores(vals):\n    """z-score over the non-missing values (ddof = 1).  Returns (z list with 0.0 for missing, covered flag)."""\n    xs = [float(v) for v in vals if _ok(v)]\n    n = len(xs)\n    if n < 2:\n        return [0.0] * len(vals), False\n    mean = sum(xs) / n\n    sd = math.sqrt(sum((x - mean) ** 2 for x in xs) / (n - 1))\n    if not sd > 1e-12:\n        return [0.0] * len(vals), False\n    return [((float(v) - mean) / sd) if _ok(v) else 0.0 for v in vals], True\n\n\ndef rerank_molecule(formulas, ranker, fwd, lams, topn=60):\n    """formulas, ranker: one entry per candidate (list order = current ranking).\n    fwd: {model: [score | None, ...]}; lams: {model: weight}.  Lists shorter than the\n    candidate list are padded with None.  Returns (perm, info)."""\n    n = len(formulas)\n    top = min(n, int(topn))\n    ranker = list(ranker) + [None] * (n - len(ranker))\n    fwd = {m: list(v) + [None] * (n - len(v)) for m, v in fwd.items()}\n    perm = list(range(n))\n    groups = {}\n    for i in range(top):\n        if formulas[i] is not None:\n            groups.setdefault(formulas[i], []).append(i)\n    info = {"groups": 0, "groups_covered": 0, "groups_changed": 0, "members_covered": 0}\n    for members in groups.values():\n        if len(members) < 2:\n            continue\n        info["groups"] += 1\n        zs, covered = {}, False\n        for m, lam in lams.items():\n            if m not in fwd or not lam:\n                continue\n            z, cov = zscores([fwd[m][i] for i in members])\n            if cov:\n                zs[m], covered = z, True\n        if not covered:\n            continue\n        info["groups_covered"] += 1\n        info["members_covered"] += len(members)\n        rs = [ranker[i] for i in members]\n        if not all(_ok(r) for r in rs):\n            rs = [-float(i) for i in members]\n        zr, _ = zscores(rs)\n        fused = []\n        for j in range(len(members)):\n            val = zr[j] + sum(lams[m] * zs[m][j] for m in zs)\n            fused.append(round(val / GRID) * GRID)\n        order = sorted(range(len(members)), key=lambda j: (-fused[j], j))\n        if order != list(range(len(members))):\n            info["groups_changed"] += 1\n        for slot, j in zip(members, order):\n            perm[slot] = members[j]\n    return perm, info\n\n\ndef list_changes(old, new, k=25):\n    """Compare two orderings of keys: top-1 changed, top-k order changed, top-k set changed."""\n    a, b = list(old[:k]), list(new[:k])\n    return {"top1": bool(a[:1] != b[:1]), "order": bool(a != b), "set": bool(set(a) != set(b))}\n', 'fm_stage.py': '"""E3b forward-model stage (ours, MIT; E3\'s stage plus library protection, see research/analysis/e3_diagnosis.md): score same-formula candidates of the engine\'s top list with\nGLACIER and ICEBERG (ms-pred, MIT, MassSpecGym weights) in subprocesses that use their own RDKit\n2025.03.x site directory, then re-order inside formula groups (fm_rerank).\n\nE3b change: a candidate whose own library similarity (``lib`` of the engine list) is >= ``protect_lib`` belongs to no\nformula group: it is not sent to the forward models and keeps its slot.  Measured evidence is never overruled by a\npredicted spectrum.\n\n``run_fm_stage`` never changes its input.  It returns (new_lists, stats); any exception is left to\nthe caller, which keeps the engine lists unchanged.\n"""\nimport glob\nimport json\nimport os\nimport shutil\nimport subprocess\nimport sys\nimport time\n\nfrom fm_rerank import list_changes, rerank_molecule\n\nADDUCTS = ("[M+H]+", "[M+Na]+")      # the only adducts in MassSpecGym, the models\' training data\nINSTRUMENT = "QTOF"                  # test spectra are all timsTOF (a Q-TOF); tokens seen in training: Orbitrap, QTOF\nMODELS = ("gl", "ice")               # run order: GLACIER (cheap) first, ICEBERG with what is left\nMODEL_NAME = {"gl": "glacier", "ice": "iceberg"}\n\n\ndef log(*a):\n    print("[fm]", *a, flush=True)\n\n\ndef formulas_of(smiles):\n    from rdkit import Chem\n    from rdkit.Chem.rdMolDescriptors import CalcMolFormula\n    from rdkit import rdBase\n    _quiet = rdBase.BlockLogs()  # noqa: F841 - silences parse errors until it goes out of scope\n    out = []\n    for s in smiles:\n        try:\n            m = Chem.MolFromSmiles(s)\n            out.append(CalcMolFormula(m) if m is not None else None)\n        except Exception:  # noqa: BLE001\n            out.append(None)\n    return out\n\n\ndef locate(root_hint=None, work=None):\n    """Find the casmi-fm-runner dataset (directory that holds code/ice_runner.py)."""\n    base = root_hint or "/kaggle/input"\n    hits = sorted(glob.glob(os.path.join(base, "**", "ice_runner.py"), recursive=True), key=len)\n    if not hits and work:  # the dataset folders are still zip archives: unpack them\n        import zipfile\n        for cz in sorted(glob.glob(os.path.join(base, "**", "code.zip"), recursive=True), key=len):\n            if any(n.endswith("ice_runner.py") for n in zipfile.ZipFile(cz).namelist()):\n                dst = os.path.join(work, "ds")\n                shutil.rmtree(dst, ignore_errors=True)\n                for z in glob.glob(os.path.join(os.path.dirname(cz), "*.zip")):\n                    zipfile.ZipFile(z).extractall(os.path.join(dst, os.path.splitext(os.path.basename(z))[0]))\n                hits = sorted(glob.glob(os.path.join(dst, "**", "ice_runner.py"), recursive=True), key=len)\n                break\n    if not hits:\n        raise FileNotFoundError("ice_runner.py (dataset casmi-fm-runner) not found")\n    code = os.path.dirname(hits[0])\n    root = os.path.dirname(code)\n\n    def one(pattern):\n        h = sorted(glob.glob(os.path.join(root, "**", pattern), recursive=True), key=len)\n        if not h:\n            raise FileNotFoundError(pattern)\n        return h[0]\n    src = os.path.dirname(os.path.dirname(one(os.path.join("ms_pred", "__init__.py"))))\n    return dict(root=root, code=code, runner=hits[0], ms_pred_src=src, gen=one("iceberg_gen.pt"),\n                inten=one("iceberg_inten.pt"), glacier=one("glacier.pt"))\n\n\ndef install_site(root, work, python_exe):\n    """RDKit 2025.03.x into its own directory (the notebook process keeps the metric\'s RDKit)."""\n    tag = "cp%d%d" % sys.version_info[:2] if python_exe == sys.executable else "cp"\n    site, extra = os.path.join(work, "site_rdkit2025"), os.path.join(work, "site_extra")\n    whl = [w for w in (glob.glob(os.path.join(root, "**", "rdkit-2025.3.*.whl"), recursive=True) + glob.glob("/kaggle/input/**/rdkit-2025.3.*-" + tag + "-*.whl", recursive=True)) if tag in os.path.basename(w)]\n    if not whl:\n        raise FileNotFoundError(f"no rdkit-2025.3.x wheel for {tag}")\n    for d, wheels in ((site, whl[:1]), (extra, sorted(glob.glob(os.path.join(root, "**", "*-py3-none-any.whl"), recursive=True)))):\n        shutil.rmtree(d, ignore_errors=True)\n        os.makedirs(d)\n        if not wheels:\n            continue\n        r = subprocess.run([python_exe, "-m", "pip", "install", "-q", "--no-index", "--no-deps", "--target", d] + wheels,\n                           capture_output=True, text=True)\n        if r.returncode != 0:\n            raise RuntimeError("pip install failed: " + r.stderr[-1500:])\n    chk = subprocess.run([python_exe, "-c", "import sys; sys.path.insert(0, sys.argv[1]); import rdkit; "\n                          "from rdkit import Chem; assert Chem.MolFromSmiles(\'CCO\') is not None; print(rdkit.__version__)", site],\n                         capture_output=True, text=True)\n    ver = chk.stdout.strip().splitlines()[-1] if chk.stdout.strip() else ""\n    if not ver.startswith("2025.03"):\n        raise RuntimeError(f"pinned RDKit not importable: {ver!r} {chk.stderr[-1500:]}")\n    return site, extra, ver\n\n\ndef select(lists, covered_mids, topn, protect_lib=None):\n    """Per molecule: formulas of the top list and the positions that share a formula with another one."""\n    sel = {}\n    for mid, e in lists.items():\n        smi = list(e.get("smiles", []))[:topn]\n        if mid not in covered_mids or len(smi) < 2:\n            continue\n        f = formulas_of(smi)\n        if protect_lib is not None:\n            lib = list(e.get("lib") or [])\n            if len(lib) < len(f):\n                raise ValueError(f"{mid}: no per-candidate library similarity in the engine list")\n            f = [None if float(lib[i]) >= protect_lib else x for i, x in enumerate(f)]\n        cnt = {}\n        for x in f:\n            if x is not None:\n                cnt[x] = cnt.get(x, 0) + 1\n        send = [i for i, x in enumerate(f) if x is not None and cnt[x] >= 2]\n        if send:\n            sel[mid] = dict(formulas=f, send=send)\n    return sel\n\n\ndef run_model(tag, paths, inp, out, test_parquet, budget, site, extra, python_exe, device, work, extra_args=()):\n    """One runner subprocess.  Returns the parsed output (possibly partial) or None."""\n    cmd = [python_exe, paths["runner"], "--input", inp, "--output", out, "--model", MODEL_NAME[tag],\n           "--spectra-parquet", test_parquet, "--ms-pred-src", paths["ms_pred_src"], "--extra-path", extra,\n           "--device", device, "--budget-sec", str(int(budget)), "--adducts", ",".join(ADDUCTS),\n           "--instrument", INSTRUMENT, "--default-ces", "20,40,60", "--save-every-sec", "60"]\n    if site:\n        cmd += ["--prepend-path", site]\n    cmd += ["--glacier-ckpt", paths["glacier"]] if tag == "gl" else ["--gen-ckpt", paths["gen"], "--inten-ckpt", paths["inten"]]\n    cmd += list(extra_args)\n    logf = os.path.join(work, f"fm_{tag}.log")\n    env = dict(os.environ, PYTHONDONTWRITEBYTECODE="1", MPLBACKEND="Agg")\n    if os.path.exists(out):\n        os.remove(out)\n    t0 = time.time()\n    try:\n        with open(logf, "w") as fh:\n            rc = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT, timeout=budget + 240, env=env, cwd=work).returncode\n    except subprocess.TimeoutExpired:\n        rc = "timeout (killed)"\n    try:\n        tail = open(logf, errors="replace").read()[-2500:]\n    except OSError:\n        tail = ""\n    log(f"{tag}: runner exit {rc} after {time.time() - t0:.0f}s (budget {budget:.0f}s); log tail:\\n{tail}")\n    try:\n        res = json.load(open(out))\n    except Exception as exc:  # noqa: BLE001\n        log(f"{tag}: NO OUTPUT ({exc!r})")\n        return None\n    res["meta"]["wall_sec"] = round(time.time() - t0, 1)\n    res["meta"]["exit"] = rc\n    return res\n\n\ndef run_fm_stage(lists, test_parquet, work, budget_sec=5400.0, topn=60, lam_ice=0.5, lam_gl=0.5, gl_share=0.4, protect_lib=0.6,\n                 root_hint=None, python_exe=None, site=None, extra=None, device="auto", extra_args=(), need_rdkit="2025.03"):\n    import pandas as pd\n    t_start = time.time()\n    python_exe = python_exe or sys.executable\n    test_parquet = os.path.abspath(test_parquet)\n    os.makedirs(work, exist_ok=True)\n    paths = locate(root_hint, work)\n    if site is None:\n        site, extra, ver = install_site(paths["root"], work, python_exe)\n        log("pinned RDKit", ver, "in", site)\n    lams = {"ice": float(lam_ice), "gl": float(lam_gl)}\n\n    te = pd.read_parquet(test_parquet, columns=["molecule_id", "adduct"])\n    covered = set(te.loc[te.adduct.astype(str).isin(ADDUCTS), "molecule_id"].astype(str))\n    sel = select(lists, covered, topn, protect_lib)\n    # weakest library match first: if the budget runs out, the well-matched molecules are the ones left unscored\n    order = sorted(sel, key=lambda m: (float(lists[m].get("lib_max") or 0.0), m))\n    inp = os.path.join(work, "fm_input.json")\n    with open(inp, "w") as fh:\n        json.dump({m: {"smiles": [lists[m]["smiles"][i] for i in sel[m]["send"]]} for m in order}, fh)\n    n_sent = sum(len(sel[m]["send"]) for m in order)\n    stats = {"n_molecules": len(lists), "n_without_covered_spectrum": sum(1 for m in lists if m not in covered),\n             "n_covered_but_no_formula_group": sum(1 for m in lists if m in covered and m not in sel),\n             "n_molecules_sent": len(order), "n_candidates_sent": n_sent, "topn": topn, "lams": lams, "protect_lib": protect_lib,\n             "n_candidates_protected": sum(1 for e in lists.values() for x in list(e.get("lib") or [])[:topn]\n                                           if protect_lib is not None and float(x) >= protect_lib),\n             "budget_sec": budget_sec, "adducts": list(ADDUCTS), "instrument": INSTRUMENT, "models": {}}\n    log(f"{len(lists)} molecules: {stats[\'n_without_covered_spectrum\']} without a covered spectrum, "\n        f"{stats[\'n_covered_but_no_formula_group\']} covered but no formula group >= 2, {len(order)} sent with {n_sent} candidates")\n\n    fwd = {}\n    for k, tag in enumerate(MODELS):\n        if not lams[tag] or not order:\n            continue\n        left = budget_sec - (time.time() - t_start) - 30.0\n        budget = min(left, gl_share * budget_sec) if k == 0 else left\n        if budget < 120:\n            log(f"{tag}: SKIPPED, only {budget:.0f}s of budget left")\n            stats["models"][tag] = {"status": "skipped_no_budget"}\n            continue\n        res = run_model(tag, paths, inp, os.path.join(work, f"fm_{tag}.json"), test_parquet, budget, site, extra,\n                        python_exe, device, work, extra_args)\n        if res is None:\n            stats["models"][tag] = {"status": "no_output"}\n            continue\n        meta = res.get("meta", {})\n        ms = {k2: meta.get(k2) for k2 in ("status", "device", "rdkit", "torch", "batch_size", "load_sec", "elapsed_sec", "wall_sec",\n                                          "n_predictions", "predict_sec", "pred_per_sec", "molecules_done", "n_molecules_covered",\n                                          "out_of_time", "reasons", "exit")}\n        if meta.get("error") or meta.get("first_predict_error"):\n            log(f"{tag}: runner error:\\n{meta.get(\'error\') or meta.get(\'first_predict_error\')}")\n            ms["error"] = (meta.get("error") or meta.get("first_predict_error"))[-600:]\n        sc = res.get("scores", {})\n        if need_rdkit and not str(meta.get("rdkit", "")).startswith(need_rdkit):\n            log(f"{tag}: SCORES DISCARDED, runner used RDKit {meta.get(\'rdkit\')!r} instead of {need_rdkit}.x")\n            ms["status"] = f"discarded_wrong_rdkit({meta.get(\'status\')})"\n            sc = {}\n        got = {}\n        for m in order:\n            v = sc.get(m)\n            if isinstance(v, list) and len(v) == len(sel[m]["send"]):\n                got[m] = [x if isinstance(x, (int, float)) else None for x in v]\n        n_scored = sum(1 for v in got.values() for x in v if x is not None)\n        mol_scored = sum(1 for v in got.values() if any(x is not None for x in v))\n        ms.update(n_candidates_scored=n_scored, n_candidates_unscored=n_sent - n_scored, n_molecules_with_scores=mol_scored,\n                  n_molecules_without_scores=len(order) - mol_scored)\n        stats["models"][tag] = ms\n        fwd[tag] = got\n        log(f"{tag}: status {ms[\'status\']}, {n_scored}/{n_sent} candidates scored in {mol_scored}/{len(order)} molecules, "\n            f"{ms.get(\'n_predictions\')} predictions, {ms.get(\'pred_per_sec\')} pred/s, wall {ms.get(\'wall_sec\')}s")\n\n    new = {}\n    ch = {"top1": 0, "order": 0, "set": 0}\n    tot = {"groups": 0, "groups_covered": 0, "groups_changed": 0, "members_covered": 0}\n    n_any = 0\n    for mid, e in lists.items():\n        new[mid] = e\n        if mid not in sel:\n            continue\n        n = len(e["smiles"])\n        f = sel[mid]["formulas"] + [None] * (n - len(sel[mid]["formulas"]))\n        per = {}\n        for tag, got in fwd.items():\n            if mid in got:\n                col = [None] * n\n                for i, x in zip(sel[mid]["send"], got[mid]):\n                    col[i] = x\n                per[tag] = col\n        if not per:\n            continue\n        perm, info = rerank_molecule(f, e.get("scores") or [], per, lams, topn)\n        if sorted(perm) != list(range(n)):\n            raise AssertionError(f"not a permutation for {mid}")\n        for k2 in tot:\n            tot[k2] += info[k2]\n        n_any += int(info["groups_covered"] > 0)\n        if perm == list(range(n)):\n            continue\n        e2 = dict(e)\n        for key in ("smiles", "keys", "scores", "lib"):\n            if isinstance(e.get(key), list) and len(e[key]) == n:\n                e2[key] = [e[key][i] for i in perm]\n        new[mid] = e2\n        c = list_changes(e["keys"], e2["keys"], 25)\n        for k2 in ch:\n            ch[k2] += int(c[k2])\n    stats.update(n_molecules_with_covered_group=n_any, formula_groups=tot, n_top1_changed=ch["top1"],\n                 n_top25_order_changed=ch["order"], n_top25_set_changed=ch["set"], stage_sec=round(time.time() - t_start, 1))\n    log(f"re-ordered: top-1 changed in {ch[\'top1\']} molecules, top-25 order changed in {ch[\'order\']}, "\n        f"top-25 membership changed in {ch[\'set\']} (of {len(lists)}); stage {stats[\'stage_sec\']}s")\n    return new, stats\n'}
for _n, _src in E3_FILES.items():
    open(os.path.join(E3_DIR, _n), 'w').write(_src)
ENG_E1 = {m: list(e.get('keys', [])) for m, e in ENG.items()}
E1_TOP25 = {m: list(dict.fromkeys(e.get('smiles', [])))[:25] for m, e in ENG.items()}
FM_STATS = {'status': 'not run'}
_rd_before = rdkit.__version__
if FM_ENABLE and len(ENG):
    try:
        _t0 = time.time()
        if E3_DIR not in sys.path:
            sys.path.insert(0, E3_DIR)
        import fm_stage
        _new, _st = fm_stage.run_fm_stage(copy.deepcopy(ENG), os.path.join(COMP, 'test.parquet'), '/tmp/fm_work',
                                          budget_sec=FM_BUDGET_SEC, topn=FM_TOPN, lam_ice=LAM_ICE, lam_gl=LAM_GL,
                                          protect_lib=FM_PROTECT_LIB)
        assert set(_new) == set(ENG), 'molecule set changed'
        for _m, _e in ENG.items():
            assert sorted(_new[_m]['keys']) == sorted(_e['keys']) and len(_new[_m]['smiles']) == len(_e['smiles']), _m
            assert len(set(_new[_m]['keys'])) == len(_new[_m]['keys']), _m
        ENG = _new
        FM_STATS = dict(_st, status='ok', total_sec=round(time.time() - _t0, 1))
    except Exception as e:
        FM_STATS = {'status': 'FAILED', 'error': repr(e)}
        print('!' * 100)
        print('FORWARD-MODEL STAGE FAILED -> E1 LISTS KEPT UNCHANGED:', repr(e))
        traceback.print_exc()
        print('!' * 100)
import rdkit as _rd
assert _rd.__version__ == _rd_before, 'the notebook RDKit changed'
for _f in sorted(glob.glob('/tmp/fm_work/fm_*')):
    try:
        import shutil; shutil.copy(_f, '/kaggle/working/' + os.path.basename(_f))
    except Exception as e:
        print('copy', _f, repr(e))
json.dump(FM_STATS, open('/kaggle/working/fm_stats.json', 'w'), indent=1)
print('notebook rdkit', _rd.__version__, f'| elapsed {time.time()-T0:.0f}s')
print('FM_STATS ' + json.dumps(FM_STATS))


In [ ]:
# E6: our PubChem channel -> PCL, merged append-only into ENG (engine top-1 kept, then engine/PubChem alternate).
# Nets: shishiradhikari11/casmi-e6-pcnets (our own FPNets trained on the competition train data only, MIT).
PCL, E6_STATS = {}, {}
try:
    _t0 = time.time()
    _budget = max(600, min(5 * 3600, 8.25 * 3600 - (time.time() - T0)))
    _py = find('e6_channel.py')
    try:   # E7: same E6 lists + per-molecule target / zlog / PubChem window dump for the Class-3 channel
        _py7 = find('e7_e6_channel.py')
    except Exception as _e:
        _py7 = None
        print('E7 channel copy not found -> e6_channel.py without the ctx dump:', repr(_e))
    _bits = os.path.join(os.path.dirname(find('coco_fp.npy')), 'fp_bits.npy')  # prvsiyan's 6,930-bit selection
    rr = subprocess.run([sys.executable, _py7 or _py, os.path.join(COMP, 'test.parquet'), '/kaggle/working/e6_pc.json',
                         '--eng-dir', ENG_DIR, '--nets', os.path.dirname(_py), '--pubchem', find('pubchem_rows.parquet'),
                         '--fp-bits', _bits, '--workers', str(os.cpu_count() or 4), '--budget', str(int(_budget))]
                        + (['--dump-ctx', '/kaggle/working/e6_ctx.pkl'] if _py7 else []),
                        capture_output=True, text=True, timeout=int(_budget) + 3600)
    print(rr.stdout[-3000:]); print(rr.stderr[-3000:])
    PCL = json.load(open('/kaggle/working/e6_pc.json'))
    E6_STATS = PCL.pop('_stats', {})
    print('E6 PubChem lists', len(PCL), f'{time.time() - _t0:.0f}s')
except Exception as e:
    print('E6 CHANNEL FAILED -> E3b lists kept:', repr(e))
n_merged = n_new = 0
for mid, e in list(ENG.items()):
    p = PCL.get(mid) or {}
    if not e.get('keys') or not p.get('keys'):
        continue
    ek, es = e['keys'], e['smiles']
    seq = [(ek[0], es[0])]
    rest_e, rest_p = list(zip(ek[1:], es[1:])), list(zip(p['keys'], p['smiles']))
    for i in range(max(len(rest_e), len(rest_p))):
        seq += rest_e[i:i + 1] + rest_p[i:i + 1]
    keys, smis = [], []
    for k, s in seq:
        if k and k not in keys:
            keys.append(k); smis.append(s)
        if len(keys) >= 40:
            break
    n_merged += 1
    n_new += len(set(keys[:25]) - set(ek[:25]))
    ENG[mid] = dict(smiles=smis, keys=keys)
print(f'E6 merge: {n_merged}/{len(ENG)} molecules | new keys in top-25 {n_new} | {time.time()-T0:.0f}s')


In [ ]:
# E7: our Class-3 channel (DEC-010) -> C3L: rr_bt = round-robin union of the biotransform and mmp_edit generators
# (shishiradhikari11/casmi-e7-c3assets), inserted at ranks 4-8 of the E6 lists; E6 ranks 1-3 untouched. Ungated.
# Any failure (exception, timeout, missing context) keeps the E6 list of that molecule / of all molecules.
E6_SNAP = {m: dict(smiles=list((v or {}).get('smiles', [])), keys=list((v or {}).get('keys', []))) for m, v in ENG.items()}
C3L, E7_STATS, E7_MERGE = {}, {}, {}
_c3out = '/kaggle/working/e7_c3.json'
try:
    _t0 = time.time()
    _c3py = find('e7_c3_channel.py')
    _left = 8.4 * 3600 - (time.time() - T0)
    _budget = int(min(2.5 * 3600, _left))
    if _budget < 600:
        raise RuntimeError(f'only {_left:.0f}s left for the C3 channel')
    for _f in ('/kaggle/working/e6_ctx.pkl', '/kaggle/working/eng_ana.json'):
        if not os.path.exists(_f):
            raise FileNotFoundError(_f)
    _cmd = [sys.executable, _c3py, os.path.join(COMP, 'test.parquet'), _c3out, '--ctx', '/kaggle/working/e6_ctx.pkl',
            '--ana', '/kaggle/working/eng_ana.json', '--assets', os.path.dirname(_c3py),
            '--coco', os.path.dirname(find('coco_fp.npy')), '--workers', str(os.cpu_count() or 4), '--budget', str(_budget)]
    try:
        rr = subprocess.run(_cmd, capture_output=True, text=True, timeout=_budget + 900)
        print(rr.stdout[-3000:]); print(rr.stderr[-3000:])
    except subprocess.TimeoutExpired as _e:
        print('E7 C3 channel timed out -> only the complete lists written so far are used:', repr(_e))
    C3L = json.load(open(_c3out))
    E7_STATS = C3L.pop('_stats', {})
    C3L.pop('_info', None)
    print('E7 C3 lists', len(C3L), '|', E7_STATS, f'| {time.time() - _t0:.0f}s')
except Exception as e:
    C3L = {}
    print('E7 C3 CHANNEL FAILED -> E6 lists kept:', repr(e))
def e7_merge(ENG, C3L, keep_top=3, n_ins=5, ref_n=25, cap=40):
    """E7SPEC (DEC-010): final list = E6 ranks 1-3 unchanged; then the first n_ins C3 (rr_bt) candidates whose metric
    key is not already in the E6 top ref_n (and not repeated); then E6 ranks 4+; de-dup on the metric key; cap entries
    (the submission cell writes the first 25). ENG = {mid: {smiles, keys}} after the E6 merge; C3L = {mid: {smiles,
    keys, ...}} from e7_c3_channel.py. Any molecule without a C3 list, or any error, keeps its E6 entry unchanged;
    a result whose first keep_top keys differ from E6's is discarded (post-condition). Returns (new ENG, stats)."""
    out = dict(ENG)
    st = dict(n_mol=len(ENG), n_c3=0, n_rows_ins=0, n_ins=0, n_err=0, n_post=0)
    for mid, e in ENG.items():
        c = C3L.get(mid) if isinstance(C3L, dict) else None
        if not c or not isinstance(e, dict) or not e.get('keys'):
            continue
        st['n_c3'] += 1
        try:
            ek, es = list(e['keys']), list(e['smiles'])
            ref = set(k for k in ek[:ref_n] if k)
            ins, seen = [], set()
            for k, s in zip(c.get('keys') or [], c.get('smiles') or []):
                if not k or not isinstance(s, str) or not s or ';' in s or k in ref or k in seen:
                    continue
                ins.append((k, s)); seen.add(k)
                if len(ins) >= n_ins:
                    break
            seq = list(zip(ek[:keep_top], es[:keep_top])) + ins + list(zip(ek[keep_top:], es[keep_top:]))
            keys, smis = [], []
            for k, s in seq:
                if k and k not in keys:
                    keys.append(k); smis.append(s)
                if len(keys) >= cap:
                    break
            if keys[:keep_top] != [k for k in ek if k][:keep_top]:
                st['n_post'] += 1
                continue
            out[mid] = dict(smiles=smis, keys=keys)
            n_new = len(set(keys[:ref_n]) - ref)
            st['n_ins'] += n_new
            st['n_rows_ins'] += int(n_new > 0)
        except Exception:
            st['n_err'] += 1
    return out, st

try:
    ENG, E7_MERGE = e7_merge(ENG, C3L)
except Exception as e:
    print('E7 MERGE FAILED -> E6 lists kept:', repr(e))
print('E7 merge:', E7_MERGE, f'| {time.time() - T0:.0f}s')


In [ ]:
sub = pd.read_csv(os.path.join(COMP, 'sample_submission.csv'))
assert len(ENG) > 0, 'engine produced no lists'
out, n_empty, n_top1, n_order, n_set = [], 0, 0, 0, 0
for mid in sub.molecule_id:
    e = ENG.get(str(mid)) or {}
    sm = list(dict.fromkeys(e.get('smiles', [])))[:25]
    old = E1_TOP25.get(str(mid), [])
    n_top1 += int(sm[:1] != old[:1]); n_order += int(sm != old); n_set += int(set(sm) != set(old))
    n_empty += int(not sm)
    out.append((mid, ';'.join(sm) if sm else 'CCO'))
fs = pd.DataFrame(out, columns=['molecule_id', 'smiles'])
assert fs.molecule_id.is_unique and fs.smiles.notna().all() and len(fs) == len(sub) and list(fs.molecule_id) == list(sub.molecule_id)
_n = fs.smiles.str.split(';').map(len); _u = fs.smiles.str.split(';').map(lambda v: len(set(v)))
assert (_n <= 25).all() and (_n == _u).all() and (fs.smiles.str.len() > 0).all() and not fs.smiles.str.contains(';;').any()
fs.to_csv('submission.csv', index=False)
pd.DataFrame([(m, ';'.join(E1_TOP25.get(str(m), [])) or 'CCO') for m in sub.molecule_id],
             columns=['molecule_id', 'smiles']).to_csv('e1_order_reference.csv', index=False)
print('E7 submission written:', len(fs), 'rows | without a list:', n_empty, '| SMILES per row min/median/max:',
      int(_n.min()), int(_n.median()), int(_n.max()), '| all unique within row:', bool((_n == _u).all()))
print('E7 vs E1 submission rows: top-1 changed', n_top1, '| top-25 order changed', n_order, '| top-25 membership changed', n_set,
      '| forward stage', FM_STATS.get('status'), f'| {time.time()-T0:.0f}s')

try:   # diagnostics only: submission.csv is already written
    _L6 = {m: list(dict.fromkeys((E6_SNAP.get(m) or {}).get('smiles', [])))[:25] for m in map(str, sub.molecule_id)}
    _L7 = {m: list(dict.fromkeys((ENG.get(m) or {}).get('smiles', [])))[:25] for m in map(str, sub.molecule_id)}
    json.dump({m: dict(smiles=v) for m, v in _L6.items()}, open('e6_lists.json', 'w'))
    json.dump({m: dict(smiles=v) for m, v in _L7.items()}, open('e7_lists.json', 'w'))
    json.dump(dict(e6=E6_STATS, e7_channel=E7_STATS, e7_merge=E7_MERGE), open('e7_stats.json', 'w'))
    print('E7 visible check: top-3 unchanged vs E6', sum(_L7[m][:3] == _L6[m][:3] for m in _L7), '/', len(_L7),
          '| 25 unique per row', sum(len(v) == 25 == len(set(v)) for v in _L7.values()), '/', len(_L7),
          '| rows with C3 insertions', sum(set(_L7[m]) != set(_L6[m]) for m in _L7))
    print('e6_lists.json / e7_lists.json written | E6 stats', E6_STATS, '| E7 channel', E7_STATS, '| E7 merge', E7_MERGE)
except Exception as e:
    print('E7 diagnostics failed (submission unaffected):', repr(e))
